# Adzuna Sector-Level Hiring Pipeline

This notebook follows the sector-level logic:

**Companies House → 8 sector keywords → Adzuna search → sector hiring summary → merge back to companies**

It does **not** rely on company website discovery, ATS slugs, or a company website map.
Instead, it uses one keyword bundle per sector to estimate hiring intensity at the sector level.

The final output assigns the same sector hiring score to all companies in the same sector.

In [ ]:
import os
import json
import time
import sqlite3
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


In [ ]:
# =========================================================
# Configuration
# =========================================================
INPUT_PATH = Path("companies_house_8_sectors_cleaned.csv")

OUTPUT_DIR = Path("adzuna_sector_pipeline_output")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_SECTOR_JOBS_CSV = OUTPUT_DIR / "adzuna_sector_jobs_raw.csv"
SECTOR_SUMMARY_CSV = OUTPUT_DIR / "adzuna_sector_hiring_summary.csv"
COMPANY_ENRICHED_CSV = OUTPUT_DIR / "companies_with_sector_hiring.csv"
CACHE_DB = OUTPUT_DIR / "adzuna_sector_cache.sqlite"

# Adzuna credentials
# Recommended: set them in environment variables.
# Example:
#   export ADZUNA_APP_ID="xxxx"
#   export ADZUNA_APP_KEY="yyyy"
ADZUNA_APP_ID = os.getenv("ADZUNA_APP_ID", "").strip()
ADZUNA_APP_KEY = os.getenv("ADZUNA_APP_KEY", "").strip()

if not ADZUNA_APP_ID or not ADZUNA_APP_KEY:
    raise ValueError("Please set ADZUNA_APP_ID and ADZUNA_APP_KEY in your environment.")

BASE_URL = "https://api.adzuna.com/v1/api/jobs"
COUNTRY = "gb"
WHERE_VALUE = "uk"

RESULTS_PER_PAGE = 50
MAX_PAGES_PER_QUERY = 2
REQUEST_SLEEP_SECONDS = 0.4

# If you want to inspect first, change this to 10 or 20
SAMPLE_SIZE = 50

SECTOR_QUERY_MAP = {
    "Manufacturing": [
        "manufacturing",
        "production",
        "factory",
        "process engineer",
        "maintenance engineer",
    ],
    "Healthcare": [
        "healthcare",
        "nurse",
        "medical",
        "care worker",
        "clinical",
    ],
    "Technology, legal & professional": [
        "software engineer",
        "developer",
        "data analyst",
        "consultant",
        "solicitor",
    ],
    "Agriculture": [
        "agriculture",
        "farm",
        "farmer",
        "horticulture",
        "agronomist",
    ],
    "Real Estate": [
        "property",
        "estate agent",
        "lettings",
        "property manager",
        "surveying",
    ],
    "Wholesale & Retail": [
        "retail",
        "sales assistant",
        "store manager",
        "wholesale",
        "merchandiser",
    ],
    "Public sector, education & charities": [
        "teacher",
        "lecturer",
        "council",
        "charity",
        "fundraising",
    ],
    "Fast growth & emerging sector": [
        "product manager",
        "data scientist",
        "machine learning",
        "growth marketing",
        "startup",
    ],
}

TARGET_SECTORS = list(SECTOR_QUERY_MAP.keys())


In [ ]:
# =========================================================
# Load data
# =========================================================
df = pd.read_csv(INPUT_PATH, dtype=str)

required_cols = {"CompanyNumber", "CompanyName", "primary_sector"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Input file missing required columns: {sorted(missing)}")

df.head(3)


In [ ]:
# =========================================================
# Helpers
# =========================================================
def normalize_sector_name(x: str) -> str:
    if pd.isna(x):
        return pd.NA
    return str(x).strip()

def dedupe_preserve_order(items):
    out = []
    seen = set()
    for x in items:
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out

def safe_int(x):
    try:
        return int(x)
    except Exception:
        return None


In [ ]:
# =========================================================
# Company subset for the first run
# =========================================================
df["primary_sector"] = df["primary_sector"].apply(normalize_sector_name)

focus_companies = df[df["primary_sector"].isin(TARGET_SECTORS)].copy()

# sample first for testing
focus_companies = (
    focus_companies[["CompanyNumber", "CompanyName", "primary_sector"]]
    .drop_duplicates()
    .head(SAMPLE_SIZE)
    .copy()
)

focus_companies.shape


In [ ]:
# =========================================================
# Requests session with retries
# =========================================================
def make_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=5,
        read=5,
        connect=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

session = make_session()


In [ ]:
# =========================================================
# SQLite cache
# =========================================================
def init_cache():
    conn = sqlite3.connect(CACHE_DB)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS sector_cache (
            cache_key TEXT PRIMARY KEY,
            response_json TEXT,
            fetched_at TEXT
        )
    """)
    conn.commit()
    conn.close()

def sector_cache_key(sector_name: str, query: str, where: Optional[str], page: int) -> str:
    return f"{sector_name}|{query.lower().strip()}|{(where or '').lower().strip()}|{page}"

def cache_get(key: str) -> Optional[dict]:
    conn = sqlite3.connect(CACHE_DB)
    cur = conn.cursor()
    cur.execute("SELECT response_json FROM sector_cache WHERE cache_key = ?", (key,))
    row = cur.fetchone()
    conn.close()
    if not row:
        return None
    try:
        return json.loads(row[0])
    except Exception:
        return None

def cache_put(key: str, response: dict):
    conn = sqlite3.connect(CACHE_DB)
    cur = conn.cursor()
    cur.execute("""
        INSERT OR REPLACE INTO sector_cache (cache_key, response_json, fetched_at)
        VALUES (?, ?, datetime('now'))
    """, (key, json.dumps(response, ensure_ascii=False)))
    conn.commit()
    conn.close()

init_cache()
print("Cache initialized.")


In [ ]:
# =========================================================
# Adzuna search
# =========================================================
def adzuna_search_jobs(query: str, where: Optional[str] = None, page: int = 1, results_per_page: int = 50) -> dict:
    url = f"{BASE_URL}/{COUNTRY}/search/{page}"
    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "what": query,
        "results_per_page": results_per_page,
        "content-type": "application/json",
    }
    if where:
        params["where"] = where

    resp = session.get(url, params=params, timeout=40)
    resp.raise_for_status()
    return resp.json()


In [ ]:
# =========================================================
# Fetch one sector/query
# =========================================================
def fetch_jobs_for_sector(
    sector_name: str,
    query: str,
    where: str = WHERE_VALUE,
    max_pages: int = MAX_PAGES_PER_QUERY,
    use_cache: bool = True,
) -> pd.DataFrame:
    rows = []

    for page in range(1, max_pages + 1):
        key = sector_cache_key(sector_name, query, where, page)
        data = cache_get(key) if use_cache else None

        if data is None:
            try:
                data = adzuna_search_jobs(
                    query=query,
                    where=where,
                    page=page,
                    results_per_page=RESULTS_PER_PAGE,
                )
                if use_cache:
                    cache_put(key, data)
            except Exception as e:
                rows.append({
                    "sector_name": sector_name,
                    "query": query,
                    "page": page,
                    "error": str(e),
                })
                break

            time.sleep(REQUEST_SLEEP_SECONDS)

        results = data.get("results", []) if isinstance(data, dict) else []
        total = data.get("count", None) if isinstance(data, dict) else None

        for item in results:
            rows.append({
                "sector_name": sector_name,
                "query": query,
                "page": page,
                "adzuna_total": total,
                "job_id": item.get("id"),
                "job_title": item.get("title"),
                "company_display_name": (item.get("company") or {}).get("display_name", ""),
                "location": (item.get("location") or {}).get("display_name"),
                "location_area": " > ".join((item.get("location") or {}).get("area", [])) if (item.get("location") or {}).get("area") else None,
                "created": item.get("created"),
                "modified": item.get("modified"),
                "redirect_url": item.get("redirect_url"),
                "salary_min": item.get("salary_min"),
                "salary_max": item.get("salary_max"),
                "contract_type": item.get("contract_type"),
                "contract_time": item.get("contract_time"),
                "category": (item.get("category") or {}).get("label"),
            })

        if len(results) < RESULTS_PER_PAGE:
            break

    return pd.DataFrame(rows)


In [ ]:
# =========================================================
# Batch run across the 8 sectors
# =========================================================
def run_sector_batch(sector_query_map: dict, where: str = WHERE_VALUE) -> pd.DataFrame:
    all_raw = []

    for sector_name, queries in sector_query_map.items():
        print(f"\n=== Sector: {sector_name} ===")

        for q in queries:
            print(f"Querying: {q}")

            df_q = fetch_jobs_for_sector(
                sector_name=sector_name,
                query=q,
                where=where,
                max_pages=MAX_PAGES_PER_QUERY,
                use_cache=True,
            )

            if not df_q.empty:
                all_raw.append(df_q)

            time.sleep(REQUEST_SLEEP_SECONDS)

    raw_df = pd.concat(all_raw, ignore_index=True) if all_raw else pd.DataFrame()
    if not raw_df.empty:
        raw_df.to_csv(RAW_SECTOR_JOBS_CSV, index=False, encoding="utf-8-sig")

    return raw_df


In [ ]:
# =========================================================
# Run sample
# =========================================================
sector_raw = run_sector_batch(SECTOR_QUERY_MAP, where=WHERE_VALUE)
print("sector_raw shape:", sector_raw.shape)
sector_raw.head(20)


In [ ]:
# =========================================================
# Deduplicate + summary
# =========================================================
def build_sector_hiring_summary(raw_df: pd.DataFrame) -> pd.DataFrame:
    if raw_df.empty:
        return pd.DataFrame()

    out = raw_df.copy()

    # avoid duplicates from multiple queries within the same sector
    out = out.drop_duplicates(subset=["sector_name", "job_id"]).copy()

    out["updated_dt"] = pd.to_datetime(out["modified"], errors="coerce", utc=True)
    out["created_dt"] = pd.to_datetime(out["created"], errors="coerce", utc=True)

    summary = (
        out.groupby("sector_name", dropna=False)
        .agg(
            vacancy_count=("job_id", lambda s: s.nunique(dropna=True)),
            unique_titles=("job_title", "nunique"),
            unique_locations=("location", "nunique"),
            latest_job_update=("updated_dt", "max"),
            earliest_job_post=("created_dt", "min"),
            query_coverage=("query", "nunique"),
        )
        .reset_index()
    )

    summary["days_since_update"] = (
        pd.Timestamp.now(tz="UTC") - summary["latest_job_update"]
    ).dt.days

    max_vacancy = max(summary["vacancy_count"].max(), 1)
    max_query_coverage = max(summary["query_coverage"].max(), 1)

    summary["volume_norm"] = summary["vacancy_count"] / max_vacancy
    summary["coverage_norm"] = summary["query_coverage"] / max_query_coverage
    summary["recency_norm"] = np.clip(1 - (summary["days_since_update"].fillna(999) / 180), 0, 1)

    # 0-100 scoring, adjustable later
    summary["sector_hiring_score"] = (
        100 * (
            0.65 * summary["volume_norm"] +
            0.20 * summary["coverage_norm"] +
            0.15 * summary["recency_norm"]
        )
    ).round(1)

    return summary


In [ ]:
sector_summary = build_sector_hiring_summary(sector_raw)
sector_summary.to_csv(SECTOR_SUMMARY_CSV, index=False, encoding="utf-8-sig")
sector_summary


In [ ]:
# =========================================================
# Merge sector hiring score back to company table
# =========================================================
company_enriched = df.merge(
    sector_summary[
        [
            "sector_name",
            "vacancy_count",
            "unique_titles",
            "unique_locations",
            "latest_job_update",
            "days_since_update",
            "sector_hiring_score",
        ]
    ],
    left_on="primary_sector",
    right_on="sector_name",
    how="left"
)

company_enriched = company_enriched.drop(columns=["sector_name"])
company_enriched["hiring_score"] = company_enriched["sector_hiring_score"]

company_enriched.head(20)


In [ ]:
# =========================================================
# Export company-level file
# =========================================================
company_enriched.to_csv(COMPANY_ENRICHED_CSV, index=False, encoding="utf-8-sig")
print(COMPANY_ENRICHED_CSV)


In [ ]:
# =========================================================
# Optional: rerun on full focus set
# =========================================================
# If the sample looks good, you can run on the full focus set:
#
# full_focus_companies = df[df["primary_sector"].isin(TARGET_SECTORS)].copy()
# full_focus_companies = full_focus_companies[["CompanyNumber", "CompanyName", "primary_sector"]].drop_duplicates()
# sector_raw_full = run_sector_batch(SECTOR_QUERY_MAP, where=WHERE_VALUE)
# sector_summary_full = build_sector_hiring_summary(sector_raw_full)
# sector_summary_full.to_csv(SECTOR_SUMMARY_CSV, index=False, encoding="utf-8-sig")
# company_enriched_full = df.merge(
#     sector_summary_full[
#         [
#             "sector_name",
#             "vacancy_count",
#             "unique_titles",
#             "unique_locations",
#             "latest_job_update",
#             "days_since_update",
#             "sector_hiring_score",
#         ]
#     ],
#     left_on="primary_sector",
#     right_on="sector_name",
#     how="left"
# ).drop(columns=["sector_name"])
# company_enriched_full["hiring_score"] = company_enriched_full["sector_hiring_score"]
# company_enriched_full.to_csv(COMPANY_ENRICHED_CSV, index=False, encoding="utf-8-sig")


## Notes

- This notebook estimates hiring pressure at the sector level.
- All companies in the same sector receive the same `sector_hiring_score`.
- The score is derived from Adzuna keyword search volume, keyword coverage, and recency.
- You can tune the sector keyword lists and the score weights later.
